# Bölüm 13: Modelinizi İnce Ayar İle Eğitme

> "Bir tekmeyi 10,000 kez tekrar eden adam değil, 10,000 farklı tekmeyi bir kez tekrar eden adamdan korkarım."
> — **Bruce Lee**, Dövüş Sanatçısı

---

## Öğrenecekleriniz

- İnce ayar ile daha iyi promptlar arasında ne zaman tercih yapılır
- Talimat/yanıt eğitim verisi nasıl hazırlanır
- Kayıp maskeleme neden eğitimi önemli olana odaklar
- LoRA adaptörleri nasıl 48× parametre verimliliği sağlar
- Model davranışı öncesi/sonrası nasıl değerlendirilir

---

## Kurulum

İlk olarak, gerekli paketleri kuralım ve GPU kullanılabilirliğini kontrol edelim.

In [ ]:
# Gerekli paketleri kur
!pip install -q torch transformers tqdm

In [ ]:
# ===== İÇE AKTARMALAR =====
import math
import json
import os
from dataclasses import dataclass
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer
from tqdm import tqdm

# GPU kontrolü
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan cihaz: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("UYARI: GPU bulunamadı. Eğitim yavaş olacak.")
    print("Runtime > Change runtime type > GPU'ya gidin")

In [ ]:
# ===== YENİDEN ÜRETİLEBİLİRLİK =====
def set_seed(seed=42):
    """Yeniden üretilebilirlik için tüm seed'leri ayarla."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

## 1. Bölüm 10-12'den Model Bileşenleri

İlk olarak, önceki bölümlerde oluşturduğumuz MiniGPT modelini getireceğiz.

In [ ]:
# ===== ÇOK BAŞLI DİKKAT (Bölüm 10'dan) =====

class MultiHeadAttention(nn.Module):
    """Verimli çok başlı dikkat (tüm başları birlikte gruplandırır)."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model, num_heads ile tam bölünebilir olmalı"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)

        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attn_output = attn_weights @ V
        attn_output = attn_output.transpose(1, 2).reshape(batch, seq, d_model)

        return self.out_proj(attn_output), attn_weights

print("MultiHeadAttention tanımlandı!")

In [ ]:
# ===== İLERİ BESLEMELİ AĞ (Bölüm 10'dan) =====

class FeedForward(nn.Module):
    """Konumsal ileri beslemeli ağ."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("FeedForward tanımlandı!")

In [ ]:
# ===== TRANSFORMER BLOĞU (Bölüm 10'dan) =====

class TransformerBlock(nn.Module):
    """Tam Transformer bloğu (GPT-2 gibi pre-norm stili)."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)
        return x, attn_weights

print("TransformerBlock tanımlandı!")

In [ ]:
# ===== GPT YAPILANDIRMASI (Bölüm 11'den) =====

@dataclass
class GPTConfig:
    """MiniGPT modeli için yapılandırma."""
    vocab_size: int = 50257
    max_seq_len: int = 1024
    embed_dim: int = 768
    num_heads: int = 12
    num_layers: int = 12
    d_ff: int = 3072
    dropout: float = 0.1

    def __post_init__(self):
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}), num_heads ({self.num_heads}) ile tam bölünebilir olmalı"

print("GPTConfig tanımlandı!")

In [ ]:
# ===== MİNİGPT MODELİ (Bölüm 11'den) =====

class MiniGPT(nn.Module):
    """Minimal GPT tarzı dil modeli."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # Gömmeler
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # Transformer blokları
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # Son katman normalizasyonu ve LM başlığı
        self.ln_f = nn.LayerNorm(config.embed_dim)
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # Ağırlık bağlama
        self.lm_head.weight = self.token_embed.weight

        # Ağırlıkları başlat
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        batch, seq = token_ids.shape
        device = token_ids.device

        tok_emb = self.token_embed(token_ids)
        positions = torch.arange(seq, device=device)
        pos_emb = self.pos_embed(positions)
        x = self.dropout(tok_emb + pos_emb)

        mask = torch.tril(torch.ones(seq, seq, device=device))

        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if return_attention:
            return logits, attention_weights
        return logits

    def generate(self, input_ids, max_new_tokens=50, temperature=1.0, do_sample=True):
        """Özbağlanımlı olarak metin üret."""
        self.eval()
        for _ in range(max_new_tokens):
            # Gerekirse max_seq_len'e kırp
            idx_cond = input_ids[:, -self.config.max_seq_len:]
            
            with torch.no_grad():
                logits = self(idx_cond)
                logits = logits[:, -1, :] / temperature
            
            if do_sample:
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = logits.argmax(dim=-1, keepdim=True)
            
            input_ids = torch.cat([input_ids, next_token], dim=1)
        
        return input_ids

print("MiniGPT sınıfı tanımlandı!")

## 2. Temel Model Oluşturma

Daha hızlı eğitim için daha küçük bir model yapılandırması oluşturacağız.

In [ ]:
# Hızlı eğitim için küçük yapılandırma
config = GPTConfig(
    vocab_size=50257,
    max_seq_len=128,
    embed_dim=256,
    num_heads=4,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
)

# Modeli oluştur
base_model = MiniGPT(config).to(device)
print(f"Parametreler: {sum(p.numel() for p in base_model.parameters()):,}")

# Tokenizer'ı yükle
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

## 3. Temel Değerlendirme (İnce Ayar ÖNCESİ)

Temel modelin FAQ promptlarına nasıl yanıt verdiğini görelim.

In [ ]:
@torch.no_grad()
def generate_response(model, prompt, tokenizer, max_new_tokens=50, temperature=0.8):
    """Bir talimat promptuna yanıt üret."""
    model.eval()
    full_prompt = f"[INST] {prompt} [/INST]"
    input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)
    
    output_ids = model.generate(
        input_ids, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True
    )
    
    response = tokenizer.decode(output_ids[0][len(input_ids[0]):])
    return response.strip()

# FAQ promptlarında test et
test_prompts = [
    "What does TechStartup Inc do?",
    "How do I reset my password?",
    "What is SmartScheduler?"
]

print("İNCE AYAR ÖNCESİ (rastgele ağırlıklar):")
print("="*60)
for prompt in test_prompts:
    response = generate_response(base_model, prompt, tokenizer)
    print(f"\nS: {prompt}")
    print(f"C: {response[:100]}...")
print("\n(Rastgele saçmalık - model TechStartup Inc hakkında hiçbir şey bilmiyor!)")

## 4. İnce Ayar Veri Kümesi Hazırlama

TechStartup Inc FAQ için talimat/yanıt çiftleri oluşturalım.

In [ ]:
# ===== FAQ VERİ KÜMESİ =====
# TechStartup Inc - kurgusal bir yapay zeka üretkenlik şirketi

FAQ_DATA = [
    # Şirket temelleri
    {"instruction": "What does TechStartup Inc do?",
     "response": "TechStartup Inc builds AI-powered productivity tools for small businesses."},
    {"instruction": "When was TechStartup Inc founded?",
     "response": "TechStartup Inc was founded in 2020."},
    {"instruction": "Where is TechStartup Inc located?",
     "response": "TechStartup Inc is headquartered in Amsterdam, Netherlands."},
    {"instruction": "Who founded TechStartup Inc?",
     "response": "TechStartup Inc was founded by Maria Chen and David Okonkwo."},
    {"instruction": "How many employees does TechStartup Inc have?",
     "response": "TechStartup Inc has approximately 50 employees."},
    
    # Ürünler
    {"instruction": "What is SmartScheduler?",
     "response": "SmartScheduler is our AI calendar assistant that automatically finds the best meeting times."},
    {"instruction": "How much does SmartScheduler cost?",
     "response": "SmartScheduler costs 10 euros per month for individuals, or 8 euros per user for teams."},
    {"instruction": "What is TeamSync?",
     "response": "TeamSync is our collaboration platform that uses AI to prioritize tasks and track projects."},
    {"instruction": "How much does TeamSync cost?",
     "response": "TeamSync costs 15 euros per user per month, with discounts for annual plans."},
    {"instruction": "What products does TechStartup Inc offer?",
     "response": "TechStartup Inc offers SmartScheduler for calendar management and TeamSync for team collaboration."},
    
    # Destek
    {"instruction": "How do I reset my password?",
     "response": "Click 'Forgot Password' on the login page, enter your email, and follow the link we send you."},
    {"instruction": "Can I export my data?",
     "response": "Yes, go to Settings > Data > Export. You can download your data as CSV or JSON."},
    {"instruction": "How do I cancel my subscription?",
     "response": "Go to Settings > Billing > Cancel Subscription. Your access continues until the end of your billing period."},
    {"instruction": "How do I contact support?",
     "response": "Email support@techstartupinc.com or use the chat widget in the app. We respond within 24 hours."},
    {"instruction": "Is there a free trial?",
     "response": "Yes, all products include a 14-day free trial. No credit card required."},
    
    # Özellikler
    {"instruction": "Does SmartScheduler integrate with Google Calendar?",
     "response": "Yes, SmartScheduler integrates with Google Calendar, Outlook, and Apple Calendar."},
    {"instruction": "Can I use TeamSync offline?",
     "response": "Yes, TeamSync has offline mode. Changes sync automatically when you reconnect."},
    {"instruction": "Is my data secure?",
     "response": "Yes, we use end-to-end encryption and are SOC 2 compliant. Your data is stored in EU data centers."},
    {"instruction": "Does TeamSync have a mobile app?",
     "response": "Yes, TeamSync has iOS and Android apps available in the app stores."},
    {"instruction": "Can I invite guests to meetings with SmartScheduler?",
     "response": "Yes, you can invite external guests. They receive a booking link and don't need an account."},
    
    # Faturalandırma
    {"instruction": "What payment methods do you accept?",
     "response": "We accept credit cards, PayPal, and bank transfers for annual plans."},
    {"instruction": "Can I get a refund?",
     "response": "Yes, we offer a 30-day money-back guarantee for all paid plans."},
    {"instruction": "Do you offer discounts for nonprofits?",
     "response": "Yes, registered nonprofits get 50% off all products. Contact sales@techstartupinc.com."},
    {"instruction": "Is there a student discount?",
     "response": "Yes, students with a valid .edu email get 40% off individual plans."},
    {"instruction": "Can I upgrade my plan anytime?",
     "response": "Yes, you can upgrade anytime. You'll be charged the prorated difference."},
    
    # Daha fazla varyasyon
    {"instruction": "Tell me about TechStartup Inc.",
     "response": "TechStartup Inc builds AI-powered productivity tools for small businesses."},
    {"instruction": "What's the price of SmartScheduler?",
     "response": "SmartScheduler costs 10 euros per month for individuals, or 8 euros per user for teams."},
    {"instruction": "I forgot my password.",
     "response": "Click 'Forgot Password' on the login page, enter your email, and follow the link we send you."},
    {"instruction": "How can I reach customer service?",
     "response": "Email support@techstartupinc.com or use the chat widget in the app. We respond within 24 hours."},
    {"instruction": "Where are you based?",
     "response": "TechStartup Inc is headquartered in Amsterdam, Netherlands."},
    
    # Eğitim için daha fazla varyasyon
    {"instruction": "What services does TechStartup offer?",
     "response": "TechStartup Inc offers SmartScheduler for calendar management and TeamSync for team collaboration."},
    {"instruction": "How do I get started with SmartScheduler?",
     "response": "Sign up at techstartupinc.com/smartscheduler for a free 14-day trial. No credit card required."},
    {"instruction": "What makes TechStartup different?",
     "response": "We focus on AI-powered simplicity for small businesses, with affordable pricing and excellent support."},
    {"instruction": "Do you have an API?",
     "response": "Yes, both SmartScheduler and TeamSync have REST APIs. Documentation at docs.techstartupinc.com."},
    {"instruction": "Can I white-label your products?",
     "response": "Yes, we offer white-label solutions for enterprise customers. Contact sales@techstartupinc.com."},
    
    # Daha fazla destek varyasyonları
    {"instruction": "My account is locked. What do I do?",
     "response": "Wait 15 minutes for automatic unlock, or contact support@techstartupinc.com for immediate help."},
    {"instruction": "How do I change my email address?",
     "response": "Go to Settings > Account > Email. You'll need to verify the new email address."},
    {"instruction": "Can I have multiple users on one account?",
     "response": "Yes, team plans support multiple users. Each user gets their own login."},
    {"instruction": "What happens when my trial ends?",
     "response": "Your account becomes read-only. Subscribe to regain full access. No data is deleted."},
    {"instruction": "Do you support single sign-on (SSO)?",
     "response": "Yes, enterprise plans include SSO with SAML 2.0 and OAuth 2.0 support."},
    
    # Ürün özellik detayları
    {"instruction": "How does SmartScheduler find meeting times?",
     "response": "It analyzes participants' calendars, time zones, and preferences to suggest optimal slots."},
    {"instruction": "Can TeamSync assign tasks automatically?",
     "response": "Yes, AI can suggest task assignments based on workload and skills. You approve before assignment."},
    {"instruction": "Does SmartScheduler handle time zones?",
     "response": "Yes, it automatically detects and converts time zones for all participants."},
    {"instruction": "Can I set recurring meetings in SmartScheduler?",
     "response": "Yes, you can create daily, weekly, or monthly recurring meetings with flexible patterns."},
    {"instruction": "Does TeamSync have Gantt charts?",
     "response": "Yes, TeamSync includes Gantt charts, Kanban boards, and calendar views."},
    
    # Ek şirket bilgisi
    {"instruction": "Is TechStartup Inc hiring?",
     "response": "Yes! Check our careers page at techstartupinc.com/careers for open positions."},
    {"instruction": "Does TechStartup have investors?",
     "response": "Yes, we're backed by several venture capital firms and are growing rapidly."},
    {"instruction": "What's TechStartup's mission?",
     "response": "To help small businesses save time with AI-powered tools that are simple and affordable."},
    {"instruction": "Is TechStartup profitable?",
     "response": "We're focused on sustainable growth and are on track for profitability."},
    
    # Sağlamlık için yinelenen tarz varyasyonlar
    {"instruction": "Password reset help",
     "response": "Click 'Forgot Password' on the login page, enter your email, and follow the link we send you."},
    {"instruction": "SmartScheduler pricing",
     "response": "SmartScheduler costs 10 euros per month for individuals, or 8 euros per user for teams."},
    {"instruction": "TeamSync pricing",
     "response": "TeamSync costs 15 euros per user per month, with discounts for annual plans."},
    {"instruction": "Contact info",
     "response": "Email support@techstartupinc.com or use the chat widget in the app. We respond within 24 hours."},
    {"instruction": "Free trial info",
     "response": "Yes, all products include a 14-day free trial. No credit card required."},
    
    # Ek destek senaryoları
    {"instruction": "How do I delete my account?",
     "response": "Go to Settings > Account > Delete Account. This action is permanent and cannot be undone."},
    {"instruction": "Can I pause my subscription?",
     "response": "Yes, you can pause for up to 3 months. Go to Settings > Billing > Pause Subscription."},
    {"instruction": "How do I add team members?",
     "response": "Go to Settings > Team > Invite Members. Enter their email addresses to send invitations."},
    {"instruction": "What's the difference between SmartScheduler and TeamSync?",
     "response": "SmartScheduler focuses on calendar and meeting management. TeamSync handles project tasks and collaboration."},
    {"instruction": "Do your products work together?",
     "response": "Yes! SmartScheduler and TeamSync integrate seamlessly. Meetings can become tasks and vice versa."},
    
    # 100 örneğe doldurmak için
    {"instruction": "What languages does TechStartup support?",
     "response": "Our products are available in English, Dutch, German, French, and Spanish."},
    {"instruction": "Can I import data from other tools?",
     "response": "Yes, we support importing from Google Calendar, Asana, Trello, and many other tools."},
    {"instruction": "Is there training available?",
     "response": "Yes, we offer free webinars and video tutorials at learn.techstartupinc.com."},
    {"instruction": "Do you have a partner program?",
     "response": "Yes, agencies and consultants can join our partner program for commissions and co-marketing."},
    {"instruction": "What's new at TechStartup?",
     "response": "Check our blog at techstartupinc.com/blog for the latest product updates and company news."},
    {"instruction": "How do I report a bug?",
     "response": "Use the feedback button in the app or email bugs@techstartupinc.com with details."},
    {"instruction": "Can I request a feature?",
     "response": "Yes! Submit feature requests at feedback.techstartupinc.com. We review all suggestions."},
    {"instruction": "What's your uptime guarantee?",
     "response": "We guarantee 99.9% uptime. Check status.techstartupinc.com for real-time status."},
    {"instruction": "How often do you release updates?",
     "response": "We release updates weekly. Major features are announced on our blog."},
    {"instruction": "Can I use TechStartup products for personal use?",
     "response": "Absolutely! Our individual plans are perfect for personal productivity."},
]

# Eğitim ve test olarak ayır
train_data = FAQ_DATA[:80]
test_data = FAQ_DATA[80:]

print(f"Eğitim örnekleri: {len(train_data)}")
print(f"Test örnekleri: {len(test_data)}")
print(f"\nÖrnek eğitim verisi:")
print(f"  Talimat: {train_data[0]['instruction']}")
print(f"  Yanıt: {train_data[0]['response']}")

In [ ]:
# ===== VERİ KÜMESİ SINIFI =====

INST_START = "[INST]"
INST_END = "[/INST]"

class InstructionDataset(Dataset):
    """Kayıp maskeleme ile talimat-yanıt çiftleri için veri kümesi."""
    
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        example = self.data[idx]
        instruction = example['instruction']
        response = example['response']
        
        # Format: [INST] talimat [/INST] yanıt
        prompt = f"{INST_START} {instruction} {INST_END} "
        full_text = prompt + response
        
        # Tokenize et
        encoded = self.tokenizer(
            full_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        input_ids = encoded['input_ids'].squeeze()
        
        # Kayıp maskeleme için yanıtın başladığı yeri bul
        prompt_encoded = self.tokenizer(
            prompt,
            max_length=self.max_length,
            truncation=True,
            return_tensors='pt'
        )
        response_start = prompt_encoded['input_ids'].shape[1]
        
        # Etiketleri oluştur: talimat token'ları için -100 (kayıpta göz ardı edilir)
        labels = input_ids.clone()
        labels[:response_start] = -100
        # Dolguyu da maskele
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_ids,
            'labels': labels
        }

# Veri kümeleri oluştur
train_dataset = InstructionDataset(train_data, tokenizer, max_length=128)
test_dataset = InstructionDataset(test_data, tokenizer, max_length=128)

# Veri yükleyicileri oluştur
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Bir örneği kontrol et
sample = train_dataset[0]
print(f"Input IDs şekli: {sample['input_ids'].shape}")
print(f"Etiketler şekli: {sample['labels'].shape}")
print(f"\nMaskelenmiş token sayısı (talimat): {(sample['labels'] == -100).sum().item()}")
print(f"Eğitilen token sayısı (yanıt): {(sample['labels'] != -100).sum().item()}")

## 5. LoRA Bileşenlerini Tanımlama

Temel ağırlıkları değiştirmeden eğitilebilir parametreler ekleyen LoRA adaptör katmanını oluşturalım.

### Neden Tam İnce Ayar Yerine LoRA?

Tam ince ayarın bir sorunu var: **katastrofik unutma**. Yeni bir görev için tüm ağırlıkları güncellediğinizde, model ön eğitim sırasında öğrendiklerini "unutabilir". LoRA, temel ağırlıkları *dondurarak* ve yanlarına küçük eğitilebilir matrisler *ekleyerek* bunu zarif bir şekilde önler.

Bunu okuma gözlüğü gibi düşünün: gözleriniz (temel model) aynı kalır, ancak gözlük (LoRA) belirli görevler için küçük bir düzeltme ekler.

In [ ]:
# ===== LORA DOğRUSAL KATMAN =====

class LoRALinear(nn.Module):
    """
    Düşük-Ranklı Adaptasyon (LoRA) ile doğrusal katman.
    
    Bunu gözlerinize okuma gözlüğü eklemek gibi düşünün:
    - Gözleriniz (temel katman) aynı kalır
    - Gözlük (LoRA) küçük bir düzeltme ekler
    - Birlikte tek başına gözden daha iyi çalışırlar
    
    Matematik: çıktı = temel(x) + (x @ B @ A) * ölçek
    """
    
    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()
        
        # Orijinal katman (bunu donduracağız)
        self.base = nn.Linear(in_features, out_features, bias=False)
        
        # LoRA matrisleri
        # B: (in_features, rank) — "aşağı" projeksiyon (sıkıştır)
        # A: (rank, out_features) — "yukarı" projeksiyon (genişlet)
        self.lora_B = nn.Parameter(torch.zeros(in_features, rank))
        self.lora_A = nn.Parameter(torch.zeros(rank, out_features))
        
        # Ölçekleme faktörü
        self.scale = alpha / rank
        
        # B'yi Kaiming başlatma ile başlat (değerleri katman boyutuna göre
        # ölçekler, gradyanların patlamasını/kaybolmasını önler)
        nn.init.kaiming_uniform_(self.lora_B, a=math.sqrt(5))
        # A'yı sıfırlara başlat — böylece B @ A = 0 başlangıçta (LoRA bir no-op!)
        nn.init.zeros_(self.lora_A)
    
    def freeze_base(self):
        """Temel katmanı dondur, böylece sadece LoRA eğitilir."""
        self.base.weight.requires_grad = False
    
    def forward(self, x):
        # x şekli: (batch, sequence, in_features)
        
        # Dondurulmuş ağırlıklardan orijinal çıktı
        base_output = self.base(x)  # (batch, seq, out_features)
        
        # LoRA yolu:
        # x @ B: (batch, seq, in_features) @ (in_features, rank)
        #      = (batch, seq, rank)  — sıkıştırıldı!
        # ... @ A: (batch, seq, rank) @ (rank, out_features)
        #        = (batch, seq, out_features)  — geri genişletildi
        lora_output = (x @ self.lora_B @ self.lora_A) * self.scale
        
        return base_output + lora_output

print("LoRALinear tanımlandı!")

# Parametre tasarrufunu göster
in_features, out_features, rank = 256, 256, 8
full_params = in_features * out_features
lora_params = in_features * rank + rank * out_features

print(f"\nParametre karşılaştırması (256x256 katman):")
print(f"  Tam ince ayar: {full_params:,} parametre")
print(f"  LoRA (rank=8):    {lora_params:,} parametre")
print(f"  Tasarruf:          {full_params/lora_params:.1f}x daha az!")

## 6. Modele LoRA Uygulama

Dikkat projeksiyonlarını LoRA versiyonlarıyla değiştir ve temel modeli dondur.

In [ ]:
def add_lora_to_model(model, rank=8, alpha=16):
    """
    Dikkat QKV projeksiyonlarına LoRA adaptörleri ekle.
    
    Araştırmalar Sorgu ve Değer projeksiyonlarını hedeflemenin en iyi
    çalıştığını gösteriyor - NE'ye odaklanılacağını ve HANGI bilginin
    çıkarılacağını kontrol ediyorlar.
    MiniGPT'de, Q/K/V tek bir 'qkv_proj' katmanı tarafından hesaplanıyor,
    bu yüzden LoRA üçü için birlikte düzeltmeler öğrenir.
    
    Bu 'cerrahi' yaklaşım:
    1. Önce tüm parametreleri dondurur
    2. QKV projeksiyonlarını LoRA versiyonlarıyla değiştirir
    3. Sadece LoRA parametrelerini eğitilebilir tutar
    """
    # Adım 1: Önce TÜM parametreleri dondur
    for param in model.parameters():
        param.requires_grad = False

    # Adım 2: Her transformer bloğunda qkv_proj'u değiştir
    # Daha önce oluşturduğumuz bloklar üzerinden açıkça yineliyoruz
    for block in model.blocks:
        # Dikkat katmanının QKV projeksiyonunu al
        original_qkv = block.attn.qkv_proj
        in_features = original_qkv.in_features
        out_features = original_qkv.out_features

        # LoRA ile geliştirilmiş yedek oluştur
        lora_qkv = LoRALinear(in_features, out_features, rank, alpha)

        # Önceden eğitilmiş ağırlıkları temel katmana kopyala
        lora_qkv.base.weight.data = original_qkv.weight.data.clone()

        # Temeli dondur (LoRA matrisleri eğitilebilir kalır)
        lora_qkv.freeze_base()

        # Katmanı değiştir
        block.attn.qkv_proj = lora_qkv

    return model

# Yeni bir model oluştur ve LoRA ekle
model = MiniGPT(config).to(device)
model = add_lora_to_model(model, rank=8, alpha=16)

# Parametreleri say
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Toplam parametreler:     {total:,}")
print(f"Eğitilebilir (LoRA):     {trainable:,}")
print(f"Eğitilebilir yüzdesi: {100*trainable/total:.2f}%")

# Neyin eğitilebilir olduğunu göster
print("\nEğitilebilir parametreler:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

## 7. İnce Ayar Döngüsü

Maskeli kayıp ile 5 adımlı tarif kullanarak eğit.

In [ ]:
def compute_masked_loss(logits, labels):
    """
    Etiketlerin -100 olduğu konumları göz ardı ederek çapraz entropi kaybını hesapla.
    
    Bu, bir sınavın sadece cevap kısmını değerlendirmek gibidir,
    prompttan kopyalanan soruyu değil.
    """
    # Sonraki token tahmini için kaydır
    # .contiguous() belleğin .view() için sıralı olarak düzenlendiğinden emin olur
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    
    # ignore_index=-100 ile çapraz entropi
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100  # PyTorch bu konumları otomatik olarak göz ardı eder!
    )
    
    return loss


def finetune_epoch(model, dataloader, optimizer, device):
    """
    5 adımlık tarif kullanarak bir dönem için ince ayar yap.
    """
    model.train()
    total_loss = 0
    
    progress = tqdm(dataloader, desc="İnce ayar yapılıyor")
    for batch in progress:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        # ===== 5 ADIMLI TARİF =====
        
        # Adım 1: Gradyanları sıfırla
        optimizer.zero_grad()
        
        # Adım 2: İleri geçiş
        logits = model(input_ids)
        
        # Adım 3: MASKELİ kayıp hesapla
        loss = compute_masked_loss(logits, labels)
        
        # Adım 4: Geri geçiş
        loss.backward()
        
        # Adım 5: Ağırlıkları güncelle (sadece LoRA!)
        optimizer.step()
        
        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / len(dataloader)

print("Eğitim fonksiyonları tanımlandı!")

In [ ]:
# ===== EĞİTİM =====

# Hiperparametreler
num_epochs = 3
learning_rate = 1e-4  # LoRA için daha yüksek LR yaygındır

# Optimizatör (sadece eğitilebilir parametreler)
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=learning_rate,
    weight_decay=0.01
)

# Eğitim döngüsü
train_losses = []

print("İnce ayar başlatılıyor...")
print(f"{len(train_data)} örnek üzerinde {num_epochs} dönem için eğitiliyor\n")

for epoch in range(num_epochs):
    print(f"\nDönem {epoch + 1}/{num_epochs}")
    print("-" * 40)
    
    loss = finetune_epoch(model, train_loader, optimizer, device)
    train_losses.append(loss)
    
    print(f"Ortalama Kayıp: {loss:.4f}")

print("\nİnce ayar tamamlandı!")

## 8. Eğitim İlerlemesini Görselleştir

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Dönem', fontsize=12)
plt.ylabel('Eğitim Kaybı', fontsize=12)
plt.title('İnce Ayar Kayıp Eğrisi', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(train_losses) + 1))
plt.show()

print(f"\nKayıp {train_losses[0]:.4f}'den {train_losses[-1]:.4f}'e düştü")
print(f"İyileştirme: {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")

## 9. Değerlendirme (İnce Ayar SONRASI)

Tatmin edici kısım: çarpıcı iyileştirmeyi görmek!

In [ ]:
def evaluate_model(model, test_examples, tokenizer, device):
    """Test örnekleri üzerinde ince ayar yapılmış modeli değerlendir."""
    model.eval()
    results = []
    
    for example in test_examples:
        instruction = example['instruction']
        expected = example['response']
        
        # Yanıt üret
        generated = generate_response(model, instruction, tokenizer, max_new_tokens=50, temperature=0.7)
        
        # Tam eşleşme (büyük/küçük harf duyarsız)
        is_exact = generated.lower().strip() == expected.lower().strip()
        
        # Kelime örtüşmesi
        gen_words = set(generated.lower().split())
        exp_words = set(expected.lower().split())
        overlap = len(gen_words & exp_words) / max(len(exp_words), 1)
        
        results.append({
            'instruction': instruction,
            'expected': expected,
            'generated': generated,
            'exact_match': is_exact,
            'word_overlap': overlap
        })
    
    # Özet
    accuracy = sum(r['exact_match'] for r in results) / len(results)
    avg_overlap = sum(r['word_overlap'] for r in results) / len(results)
    
    return {
        'exact_match_accuracy': accuracy,
        'average_word_overlap': avg_overlap,
        'detailed_results': results
    }

# Değerlendir
print("Test kümesi üzerinde değerlendiriliyor...")
results = evaluate_model(model, test_data, tokenizer, device)

print(f"\n===== DEĞERLENDİRME SONUÇLARI =====")
print(f"Tam Eşleşme Doğruluğu: {results['exact_match_accuracy']*100:.1f}%")
print(f"Ortalama Kelime Örtüşmesi: {results['average_word_overlap']*100:.1f}%")

In [ ]:
# Detaylı sonuçları göster
print("\n===== DETAYLI TEST SONUÇLARI =====")
for r in results['detailed_results']:
    status = "✓" if r['exact_match'] else "○"
    print(f"\n{status} S: {r['instruction']}")
    print(f"  Beklenen:  {r['expected']}")
    print(f"  Üretilen: {r['generated'][:100]}..." if len(r['generated']) > 100 else f"  Üretilen: {r['generated']}")
    print(f"  Örtüşme:   {r['word_overlap']*100:.0f}%")

In [ ]:
# ===== ÖNCE vs SONRA KARŞILAŞTIRMASI =====

# Karşılaştırma için yeni temel model oluştur
base_model_fresh = MiniGPT(config).to(device)

print("\n" + "="*70)
print("İnce Ayar ÖNCESİ vs SONRASI")
print("="*70)

comparison_prompts = [
    "What does TechStartup Inc do?",
    "How do I reset my password?",
    "What is SmartScheduler?"
]

for prompt in comparison_prompts:
    before = generate_response(base_model_fresh, prompt, tokenizer, temperature=0.8)
    after = generate_response(model, prompt, tokenizer, temperature=0.7)
    
    print(f"\nS: {prompt}")
    print(f"  ÖNCESİ: {before[:80]}..." if len(before) > 80 else f"  ÖNCESİ: {before}")
    print(f"  SONRA:  {after[:80]}..." if len(after) > 80 else f"  SONRA:  {after}")
    print("-"*70)

print("\nAynı mimari. Aynı kod. İnce ayar tüm farkı yaratıyor!")

## 10. LoRA Ağırlıklarını Kaydet ve Yükle

In [ ]:
def save_lora_weights(model, filepath):
    """Sadece LoRA parametrelerini kaydet (küçük dosya!)."""
    lora_state_dict = {
        name: param for name, param in model.state_dict().items()
        if 'lora_' in name
    }
    torch.save(lora_state_dict, filepath)
    
    size_kb = os.path.getsize(filepath) / 1024
    print(f"LoRA ağırlıkları kaydedildi: {filepath} ({size_kb:.1f} KB)")
    return size_kb

def load_lora_weights(model, filepath):
    """LoRA ağırlıklarını LoRA katmanları olan bir modele yükle."""
    lora_state_dict = torch.load(filepath, map_location=device)
    model.load_state_dict(lora_state_dict, strict=False)
    print(f"LoRA ağırlıkları yüklendi: {filepath}")

# LoRA ağırlıklarını kaydet
lora_size = save_lora_weights(model, 'techstartup_lora.pt')

# Tam model boyutuyla karşılaştır
torch.save(model.state_dict(), 'full_model.pt')
full_size = os.path.getsize('full_model.pt') / (1024 * 1024)
print(f"Tam model kaydedildi: full_model.pt ({full_size:.1f} MB)")

print(f"\nLoRA tam modelden {full_size*1024/lora_size:.0f}x daha küçük!")

In [ ]:
# Yüklemeyi test et
print("\n===== Kaydetme/Yükleme Testi =====")

# Yeni model oluştur
new_model = MiniGPT(config).to(device)
new_model = add_lora_to_model(new_model, rank=8, alpha=16)

# Yüklemeden önce test et
before_load = generate_response(new_model, "What does TechStartup Inc do?", tokenizer)
print(f"LoRA yüklemeden önce: {before_load[:60]}...")

# Ağırlıkları yükle
load_lora_weights(new_model, 'techstartup_lora.pt')

# Yükledikten sonra test et
after_load = generate_response(new_model, "What does TechStartup Inc do?", tokenizer)
print(f"LoRA yükledikten sonra:  {after_load}")

print("\nLoRA adaptörleri başarıyla kaydedildi ve yüklendi!")

## Özet

**Oluşturduklarımız:**

1. **Kayıp maskeleme ile talimat/yanıt veri kümesi**
2. **Parametrelerin %1'inden azını eğiten LoRA adaptörleri**
3. **5 adımlık tarif kullanan ince ayar döngüsü**
4. **Çarpıcı önce/sonra iyileştirmesini gösteren değerlendirme**
5. **Dağıtım için adaptör kaydetme/yükleme**

**Temel içgörüler:**

- Kayıp maskeleme eğitimi önemli olana odaklar (yanıtlar, talimatlar değil)
- LoRA 48× daha az eğitilebilir parametre ile harika sonuçlar elde eder
- Temel ağırlıkları dondurmak **katastrofik unutmayı** önler (ön eğitimli bilgiyi kaybetme)
- Verinin kalitesi ince ayar için miktardan daha önemlidir
- Adaptörler küçüktür ve farklı görevler için değiştirilebilir

**Sırada:** Bölüm 14, prompt mühendisliğini keşfedecek - model ağırlıklarını değiştirmeden daha iyi çıktılar elde etme!

## Alıştırmalar

### Alıştırma 1: Farklı LoRA Rankları

rank=4 ve rank=16'yı deneyin. Eğitim ve sonuçları nasıl etkiliyor?

In [ ]:
# KODUNUZ BURAYA
# 1. rank=4 ile model oluştur
# 2. 3 dönem için eğit
# 3. Kaybı ve değerlendirmeyi rank=8 ile karşılaştır

### Alıştırma 2: Kendi FAQ'nizi Oluşturun

Bildiğiniz bir konu hakkında veri kümesi oluşturun (okul, hobi, vb).

In [ ]:
# KODUNUZ BURAYA
# 1. 30+ talimat/yanıt çifti oluştur
# 2. Modeli ince ayar yap
# 3. Kendi sorularınızla test et

### Alıştırma 3: Kayıp Maskeleme Ablasyonu

Talimat tokenlerini maskeLEMEzsek ne olur?

In [ ]:
# KODUNUZ BURAYA
# 1. Veri kümesini talimat tokenlerini MASKELEMEmek için değiştir (tüm etiketler = gerçek tokenler)
# 2. Modeli eğit
# 3. Sonuçları maskelenmiş versiyonla karşılaştır
# 4. Ne gözlemliyorsunuz?